# DNAffinity: train and test models

Minimal example of the DNAffinity pipeline: load a preprocessed affinity table, build the physics-based features, train a model, and look at the predictions.

It uses the bundled **Gata4 uPBM** test case. Change `REGRESSOR` below to try any model from `dnaffinity/models/`: `random_forest`, `linear`, `ridge`, `svr`, `DFN`, `CNN1D`, `CNNAttention`, `Transformer`.

In [ ]:
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/PieterZanders/DNAffinity.git
    %cd DNAffinity
else:
    pass

In [ ]:
# Setup: make the dnaffinity package importable from the repo root
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dnaffinity import Dataset, Model
from dnaffinity.common import default_params
from dnaffinity.plotting import (plot_feature_importances, plot_predictions,
                                 plot_training_curve)

np.random.seed(20)

print(f'Repo root: {REPO_ROOT}')

## 1. Load the data

`EXP_PROTOCOL` selects the experimental technique (each with its own preprocessing, mirroring `scripts/run_*.py`):
- **uPBM** — preprocessed probes (`ID_REF`), affinity (`VALUE`) and a per-probe sample weight (`WEIGHT`).
- **SELEX** — k-mer table for one cycle (`Kmer`, `Affinity`, `Probability`, ...); only the top 10% of k-mers by probability are kept; no sample weights.
- **gcPBM** — raw probe intensities; sequences are trimmed to the central 30 bp, intensities min–max normalized, and probes with secondary binding sites in the flanks (scored against the position frequency matrix) are discarded; no sample weights.

In [ ]:
EXP_PROTOCOL = 'SELEX'   # 'uPBM', 'SELEX' or 'gcPBM'
PROTEIN = {'uPBM': 'Gata4', 'SELEX': 'Gata4', 'gcPBM': 'cbf1'}[EXP_PROTOCOL]

# data lives in data/{protocol}/{protein}/ (lowercase folder names)
data_dir = REPO_ROOT / 'data' / EXP_PROTOCOL.lower() / PROTEIN

if EXP_PROTOCOL == 'uPBM':
    df_train = pd.read_csv(data_dir / f'{PROTEIN}_training.txt', sep='\t')
    WEIGHTED = True

elif EXP_PROTOCOL == 'SELEX':
    CYCLE = 4
    deca = pd.read_csv(data_dir / f'{CYCLE}.txt', sep='\t')
    # undersample: keep the top 10% of k-mers by probability, then shuffle
    cut = deca['Probability'].quantile(0.9)
    df_train = (deca.loc[deca['Probability'] > cut, ['Kmer', 'Affinity']]
                .rename(columns={'Kmer': 'ID_REF', 'Affinity': 'VALUE'})
                .sample(frac=1.0, random_state=20)
                .reset_index(drop=True))
    WEIGHTED = False

elif EXP_PROTOCOL == 'gcPBM':
    CONCENTRATION = 100
    raw = (pd.read_csv(data_dir / f'{PROTEIN}_{CONCENTRATION}.txt', sep='\t')
           [['Sequence', 'Intensity']].dropna().reset_index(drop=True))
    strings = [s[3:33] for s in raw['Sequence']]                  # central 30 bp
    values = (raw['Intensity'] - raw['Intensity'].min()) / (raw['Intensity'].max() - raw['Intensity'].min())

    # discard probes whose flanks score high against the frequency matrix (secondary binding sites)
    freq = np.array(pd.read_csv(data_dir / f'{PROTEIN}_freq_matrix_6.txt', sep=r'\s+'))
    tr = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    flanks = list(range(0, 12 - 5)) + list(range(18, 30 - 5))     # positions outside the central site

    def has_secondary_site(seq):
        return any(sum(freq[j][tr[seq[i + j]]] for j in range(len(freq))) > 2.5 for i in flanks)

    keep = np.array([not has_secondary_site(s) for s in strings])
    df_train = pd.DataFrame({'ID_REF': strings, 'VALUE': values})[keep].reset_index(drop=True)
    print(f'discarded {100 * (1 - keep.mean()):.1f}% of probes (secondary binding sites)')
    WEIGHTED = False

print(f'{len(df_train)} sequences of length {len(df_train.ID_REF.iloc[0])}')
df_train.head()

5511 probes


,ID_REF,VALUE,WEIGHT
0,ACTCAAAGAC,0.094644,1
1,ACCTATAAAG,0.070668,1
2,CCCGATAACA,0.342472,1
3,ATATCTAAGG,0.217709,1
4,TCTGAGACGG,0.061319,1


## 2. Build features and the model

`Dataset` turns the probe sequences into physics-based features. The feature blocks follow the experimental method (same as `scripts/run_*.py`): all protocols use tetramer presence, average structural parameters and force-constant diagonals; **gcPBM additionally includes electrostatic features** (5 values per tetramer window). `Model` picks the right input (flat features or per-window tensors) and builds the regressor, but does **not** train it yet — `print(model)` shows the split and the architecture.

For the PyTorch models, hyperparameters come from `default_params(REGRESSOR)` — the same defaults as `configs/{model}_config.json`. Override any of them in `params` below (e.g. fewer epochs for a quick notebook run).

In [ ]:
REGRESSOR = 'random_forest'   # try: 'DFN', 'CNN1D', 'CNNAttention', 'Transformer', 'ridge', ...

len_aln = len(df_train['ID_REF'].iloc[0])

# feature blocks depend on the experimental method (see scripts/run_*.py)
chosen_features = ['presence_tetramer', 'avg', 'diagonal_fce']
if EXP_PROTOCOL == 'gcPBM':
    chosen_features.append('electrostatic')

dataset = Dataset(
    PROTEIN,
    df_train,
    model_target='octamers',
    chosen_features=chosen_features,
    selected_tetramers=list(range(len_aln - 3)),
    weighted=WEIGHTED,
)

params = default_params(REGRESSOR)
params.update({'num_epochs': 20})   # keep PyTorch runs short in the notebook

model = Model(dataset, len_aln, regressor=REGRESSOR, training_set_size=0.9,
              weighted=WEIGHTED, pytorch_model_params=params)


### Input Feature Analysis


The feature matrix the model actually sees (`dataset.features`). With named columns in a `DataFrame`, pandas does the heavy lifting: `describe()`, histograms and correlations.

Column layout for `['presence_tetramer', 'avg', 'diagonal_fce']`: 256 tetramer counts (`presence_*`), then 6 average helical parameters per tetramer window (`avg_{param}_w{i}`), then 6 stiffness diagonals per window (`fce_{param}_w{i}`).


In [ ]:
from itertools import product

HELICAL = ['shift', 'slide', 'rise', 'tilt', 'roll', 'twist']
n_windows = len(dataset.selected_tetramers)

feature_names = (
    [f'presence_{"".join(t)}' for t in product('ACTG', repeat=4)]        # Dataset.presence() order
    + [f'avg_{p}_w{w}' for w in range(n_windows) for p in HELICAL]
    + [f'fce_{p}_w{w}' for w in range(n_windows) for p in HELICAL]
)
if 'electrostatic' in dataset.feats:   # gcPBM: 5 electrostatic values per window
    feature_names += [f'elec_{k}_w{w}' for w in range(n_windows) for k in range(5)]

feat_df = pd.DataFrame(dataset.features.astype(float), columns=feature_names)
feat_df['target'] = dataset.scores.ravel()
print(f'feature matrix: {feat_df.shape[0]} probes x {feat_df.shape[1] - 1} features')



#### Summary statistics of the physics-based features 

Value ranges differ a lot between blocks (counts vs structural parameters vs force constants) 
this is why the PyTorch models apply a LayerNorm on the input.

In [ ]:
feat_df.filter(regex='^(avg|fce)_').describe().T.round(3)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
tetramer_counts = feat_df.filter(regex='^presence_').sum().sort_values(ascending=False)
tetramer_counts.head(15).rename(lambda c: c.replace('presence_', '')).plot.bar(ax=axes[0])
axes[0].set_ylabel('occurrences in library')
axes[0].set_xlabel('most frequent tetramers')
feat_df.filter(regex='^avg_').stack().hist(bins=80, ax=axes[1])
axes[1].set_xlabel('avg helical parameters (all windows)')
feat_df.filter(regex='^fce_').stack().hist(bins=80, ax=axes[2])
axes[2].set_xlabel('stiffness diagonals (all windows)')
plt.tight_layout()
plt.show()

#### Full Feature decomposition

In [ ]:
avg_cols = feat_df.filter(regex='^avg_')
n_rows = -(-avg_cols.shape[1] // 6)   # ceil; adapts to the number of windows
avg_cols.hist(bins=100, figsize=(16, 1.8 * n_rows), layout=(n_rows, 6))
plt.tight_layout()

#### Correlation with the target

Single-feature linear correlation with the scaled affinity — a quick, model-free look at which features carry signal (compare with the random forest importances further down).


In [ ]:
corr = feat_df.corr(numeric_only=True)['target'].drop('target')
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(15)

ax = top.sort_values().plot.barh(figsize=(7, 5))
ax.set_xlabel('Pearson correlation with target')
plt.tight_layout()
top.to_frame('correlation')
plt.show()

## 3. Train

Splits train/test and fits the regressor. PyTorch models show a progress bar with the training loss and the loss on the held-out split (monitoring only, it is not used for fitting).

In [ ]:
model.train()

### Training curve

Per-epoch loss for the PyTorch models; `model.history` is `None` for the scikit-learn models.

In [ ]:
if model.history is not None:
    plot_training_curve(model.history)
else:
    print(f'{REGRESSOR} is not trained by epochs, no training curve')

## 4. Test

`predict()` evaluates on the held-out test split (and on the training split for comparison). The scatter shows predicted vs measured affinity.

In [ ]:
model.predict()
print(f'test r2  = {model.r2}')
print(f'train r2 = {model.r2_train}')

plot_predictions(model.y_test, model.y_pred, model.r2,
                 y_train=model.y_train, y_train_pred=model.y_train_pred,
                 r2_train=model.r2_train)

## 5. Feature importances

Only available for models that expose them (random forest). For other regressors this cell prints a note and moves on.

In [ ]:
importances = plot_feature_importances(model.features, model.l)
if importances is not None:
    display(importances.head(10))
else:
    print(f'{REGRESSOR} does not provide feature importances')

## 6. Layer-wise relevance propagation (DFN / CNN1D)

ε-LRP walks the trained network backwards and redistributes each prediction onto the input features, giving the neural models an importance view analogous to the random forest importances above. It is implemented for the two architectures with well-defined LRP rules: `DFN` (flat features) and `CNN1D` (per-window features) — see `dnaffinity/utils/lrp.py`.

If the model trained above is one of those it is reused; otherwise a short DFN is trained here for the demo.

In [ ]:
from dnaffinity.utils.lrp import lrp_relevance
from dnaffinity.plotting import plot_relevance

if model.regressor_name in ('DFN', 'CNN1D'):
    lrp_model = model
else:
    dfn_params = default_params('DFN')
    dfn_params['num_epochs'] = 20
    lrp_model = Model(dataset, len_aln, regressor='DFN', training_set_size=0.9,
                      weighted=WEIGHTED, pytorch_model_params=dfn_params).train()

relevance = lrp_relevance(lrp_model)   # defaults to the held-out test split
print('relevance matrix:', relevance.shape)

Mean |relevance| per feature. For the DFN this is a bar chart over the flat features (plus a roll-up by feature block); for CNN1D it is a heatmap of tetramer windows × per-window features.

In [ ]:
if relevance.ndim == 2:   # DFN: flat features
    rel = plot_relevance(relevance, feature_names=feature_names, top=15)
    blocks = rel.groupby(rel.index.str.split('_').str[0]).sum()
    display(blocks.sort_values(ascending=False).to_frame('total mean |relevance|'))
else:                      # CNN1D: windows x per-window features
    window_names = [f'avg_{p}' for p in HELICAL] + [f'fce_{p}' for p in HELICAL]
    if 'electrostatic' in dataset.feats:
        window_names += [f'elec_{k}' for k in range(5)]
    rel = plot_relevance(relevance, feature_names=window_names)

## 7. Compare several models

Quick comparison on the same train/test split. PyTorch models run with few epochs here, so their scores are indicative only — use the scripts in `scripts/` with the full configs for real runs.

In [ ]:
results = {}
for name in ['random_forest', 'ridge', 'DFN']:
    params = default_params(name)
    params.update({'num_epochs': 20})
    m = Model(dataset, len_aln, regressor=name, training_set_size=0.9,
              weighted=WEIGHTED, pytorch_model_params=params)
    m.train()
    m.predict()
    results[name] = {'test r2': m.r2, 'train r2': m.r2_train}

pd.DataFrame(results).T